# LLAMBO-RL: LLM-Augmented Bayesian Transition Optimization

This notebook implements the LLAMBO-RL training loop on `LunarLander-v3`. A Deep
Q-Network alternates between online interaction with the real environment and
offline updates on a buffer augmented with high-return transitions *dreamed* by an
LLM acting as a joint candidate sampler and reward surrogate.

See `README.md` for the full method description. The LLM API key is read from the
`XAI_API_KEY` environment variable; set it before running:

```bash
export XAI_API_KEY="your-key"
```

In [ ]:
import os
import ast
import re
import json
import time
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI
from stable_baselines3 import DQN
from stable_baselines3.common.buffers import ReplayBuffer
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback

## LLM interface

A thin wrapper around the xAI chat API and a robust parser that recovers a
dictionary from the model's response even when it is wrapped in markdown or prose.

In [ ]:
# The API key is read from the environment to avoid committing secrets.
client = OpenAI(
    api_key=os.environ.get("XAI_API_KEY"),
    base_url="https://api.x.ai/v1",
)

LLM_MODEL = "grok-4-fast-non-reasoning"


def query_llm(system_prompt, user_prompt):
    """Send a system/user prompt pair to the LLM and return its text response."""
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.7,
            response_format={"type": "json_object"},
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"LLM API error: {e}")
        return None


def parse_llm_dict(response_text):
    """Extract the first valid dictionary from an LLM response.

    Handles responses wrapped in markdown code fences or surrounded by prose.
    Returns None if no parseable dictionary is found.
    """
    if not response_text:
        return None

    candidates = []
    # Prefer an explicit ```json ... ``` / ```python ... ``` block if present.
    fenced = re.search(r"```(?:json|python)?\s*(\{.*?\})\s*```", response_text, re.DOTALL)
    if fenced:
        candidates.append(fenced.group(1))

    # Fall back to substrings spanning the first '{' and each subsequent '}'.
    start = response_text.find("{")
    if start != -1:
        candidates.extend(
            response_text[start : end + 1]
            for end, ch in enumerate(response_text[start:], start)
            if ch == "}"
        )

    for candidate in candidates:
        for loader in (json.loads, ast.literal_eval):
            try:
                return loader(candidate)
            except (ValueError, SyntaxError):
                continue
    return None

## Dream generation and evaluation

`generate_dreams` prompts the LLM with recent real transitions and asks for new
high-return candidates; `evaluate_dreams` then has the LLM score each one, acting
as a reward surrogate.

In [ ]:
def generate_dreams(hist_data_path="datasets/historical_data.csv",
                    num_historical=50, num_samples_generated=10):
    """Prompt the LLM to imagine new high-reward transitions.

    The most recent ``num_historical`` real transitions are shown as few-shot
    examples; the LLM returns ``num_samples_generated`` new (s, a, s') candidates.
    """
    if not os.path.exists(hist_data_path):
        print(f"Historical data not found: {hist_data_path}")
        return {}

    hist_df = pd.read_csv(hist_data_path)
    recent = hist_df.tail(num_historical)[["obs", "action", "reward", "next_obs"]].values.tolist()

    examples = []
    for i, (s, a, _r, ns) in enumerate(recent, start=1):
        s_val = ast.literal_eval(s) if isinstance(s, str) else s
        ns_val = ast.literal_eval(ns) if isinstance(ns, str) else ns
        # Unwrap a single nesting level (e.g. [[...]] -> [...]) and round for brevity.
        s_flat = s_val[0] if isinstance(s_val[0], list) else s_val
        ns_flat = ns_val[0] if isinstance(ns_val[0], list) else ns_val
        s_rd = [round(float(x), 3) for x in s_flat]
        ns_rd = [round(float(x), 3) for x in ns_flat]
        examples.append(f"{i}: [{s_rd}, {a}, {ns_rd}]")
    examples_str = "\n".join(examples)

    system_prompt = (
        "LunarLander-v3 transition generator. Output ONLY a valid JSON dictionary "
        "of the form {id: [s, a, ns]}. No prose, no markdown."
    )
    user_prompt = f"""
    Reference transitions (s, a, ns):
    {examples_str}

    Goal: Generate {num_samples_generated} new realistic transitions.
    Action map: 0=None, 1=Left, 2=Main, 3=Right.
    State: [x, y, vx, vy, angle, v_angle, leg_l, leg_r]

    Output: {{1: [s_1, a_1, ns_1], ..., {num_samples_generated}: [s_n, a_n, ns_n]}}
    """
    return parse_llm_dict(query_llm(system_prompt, user_prompt))


def evaluate_dreams(dream_dict):
    """Ask the LLM to act as a reward oracle for each imagined transition."""
    if not dream_dict:
        return {}

    system_prompt = (
        "LunarLander-v3 reward oracle. Output ONLY a valid JSON dictionary of the "
        "form {id: reward} with numeric rewards. No prose, no markdown."
    )
    user_prompt = f"""
    Score the following {len(dream_dict)} LunarLander-v3 transitions.

    Reward guidance:
    1. Proximity to the landing pad at (0, 0): closer is better.
    2. Velocity: lower speed is better.
    3. Angle: closer to 0 is better.
    4. Legs: +10 per ground contact.
    5. Engine cost: main = -0.3, side = -0.03 per firing.
    Do not simply copy numbers from the state vector.

    Action map: 0=None, 1=Left, 2=Main, 3=Right.
    State: [x, y, vx, vy, angle, v_angle, leg_l, leg_r]

    Transitions:
    {dream_dict}

    Output: {{1: r_1, 2: r_2, ..., {len(dream_dict)}: r_n}}
    """
    return parse_llm_dict(query_llm(system_prompt, user_prompt))

## Environment and replay-buffer utilities

In [ ]:
vec_env = make_vec_env("LunarLander-v3", n_envs=1)


def populate_buffer(buffer, df):
    """Add transitions from a dataframe into a Stable-Baselines3 replay buffer."""
    if df.empty:
        return 0

    valid_count = 0
    for _, row in df.iterrows():
        try:
            s = ast.literal_eval(row["obs"]) if isinstance(row["obs"], str) else row["obs"]
            ns = ast.literal_eval(row["next_obs"]) if isinstance(row["next_obs"], str) else row["next_obs"]

            # Unwrap a single nesting level: [[...]] -> [...].
            if isinstance(s, list) and len(s) == 1 and isinstance(s[0], list):
                s = s[0]
            if isinstance(ns, list) and len(ns) == 1 and isinstance(ns[0], list):
                ns = ns[0]

            if len(s) != 8 or len(ns) != 8:
                print(f"Skipping transition with wrong dimensions: s={len(s)}, ns={len(ns)}")
                continue

            buffer.add(
                np.array(s, dtype=np.float32),
                np.array(ns, dtype=np.float32),
                np.array([int(row["action"])], dtype=np.int64),
                float(row["reward"]),
                bool(row["done"]),
                [{}],
            )
            valid_count += 1
        except Exception as e:
            print(f"Skipping unparseable row: {e}")

    print(f"Added {valid_count} transitions to the replay buffer.")
    return valid_count

## Logging callback

Records every real interaction (for the historical CSV) and every episode return
(for the learning-curve plot).

In [ ]:
class LoggerCallback(BaseCallback):
    """Records real environment transitions and episode returns during training."""

    def __init__(self, hist_csv_path, results_path="datasets/results.csv", verbose=0):
        super().__init__(verbose)
        self.hist_csv_path = hist_csv_path
        self.results_path = results_path
        self.step_data = []
        self.results = []

    def _on_step(self) -> bool:
        obs = self.model._last_obs
        action = self.locals["actions"][0]
        reward = self.locals["rewards"][0]
        next_obs = self.locals["new_obs"][0]
        done = self.locals["dones"][0]
        self.step_data.append(
            [obs.tolist(), int(action), float(reward), next_obs.tolist(), bool(done)]
        )

        info = self.locals["infos"][0]
        if "episode" in info:
            ep_return = info["episode"]["r"]
            self.results.append({"steps": self.model.num_timesteps, "return": ep_return})
            print(f"Step {self.model.num_timesteps} | Episode return: {ep_return:.2f}")
        return True

    def save_to_csv(self):
        df_hist = pd.DataFrame(
            self.step_data, columns=["obs", "action", "reward", "next_obs", "done"]
        )
        df_hist.to_csv(self.hist_csv_path, mode="a", index=False,
                       header=not os.path.isfile(self.hist_csv_path))

        df_res = pd.DataFrame(self.results)
        df_res.to_csv(self.results_path, mode="a", index=False,
                      header=not os.path.isfile(self.results_path))

        self.step_data = []
        self.results = []

## Offline training routines

`train_historical` is the baseline update (real data only); `train_on_dreams`
trains on real data augmented with the imagined transitions.

In [ ]:
def train_historical(model, hist_data_path, gradient_steps=100, batch_size=64):
    """Baseline offline update: train only on real historical transitions."""
    if not os.path.exists(hist_data_path):
        return
    hist_df = pd.read_csv(hist_data_path)
    populate_buffer(model.replay_buffer, hist_df)
    print(f"Training on historical data: {gradient_steps} gradient steps.")
    model.train(gradient_steps=gradient_steps, batch_size=batch_size)


def train_on_dreams(model, hist_data_path, gradient_steps=100, batch_size=64):
    """LLAMBO offline update: train on real history augmented with imagined dreams."""
    mixed_buffer = ReplayBuffer(
        buffer_size=100_000,
        observation_space=model.observation_space,
        action_space=model.action_space,
        device=model.device,
    )
    populate_buffer(mixed_buffer, pd.read_csv(hist_data_path))
    populate_buffer(mixed_buffer, pd.read_csv("datasets/imagined_data.csv"))

    # Temporarily swap in the augmented buffer, then restore the original.
    original_buffer = model.replay_buffer
    model.replay_buffer = mixed_buffer
    try:
        model.train(gradient_steps=gradient_steps, batch_size=batch_size)
    finally:
        model.replay_buffer = original_buffer


def append_dreams(transitions, rewards, top_k=5, path="datasets/imagined_data.csv"):
    """Select the top-k transitions by predicted reward and append them to disk."""
    def reward_value(key):
        r = rewards[key]
        return sum(r) if isinstance(r, list) else r

    top_keys = sorted(rewards, key=reward_value, reverse=True)[:top_k]

    data = []
    for key in top_keys:
        if key not in transitions:
            continue
        s, a, ns = transitions[key]
        data.append([s, a, reward_value(key), ns, False])

    if not data:
        print("No dreams to append.")
        return

    df = pd.DataFrame(data, columns=["obs", "action", "reward", "next_obs", "done"])
    df.to_csv(path, mode="a", index=False, header=not os.path.isfile(path))
    print(f"Top-{top_k} selection: appended {len(df)} imagined transitions.")

## Plotting and data management

In [ ]:
def plot_learning_comparison(file_configs, window=10, output_name="learning_curve.png"):
    """Plot smoothed episode-return learning curves for each experiment."""
    plt.figure(figsize=(10, 6))
    plt.style.use("ggplot")

    plotted = False
    for config in file_configs:
        if not os.path.exists(config["path"]):
            print(f"File not found: {config['path']}")
            continue

        df = pd.read_csv(config["path"])
        # Coerce to numeric and drop any repeated header rows from appended writes.
        df["steps"] = pd.to_numeric(df["steps"], errors="coerce")
        df["return"] = pd.to_numeric(df["return"], errors="coerce")
        df = df.dropna(subset=["steps", "return"]).sort_values("steps")
        if df.empty:
            print(f"No valid data in {config['path']}.")
            continue

        mean = df["return"].rolling(window=window, min_periods=1).mean()
        std = df["return"].rolling(window=window, min_periods=1).std().fillna(0)
        plt.plot(df["steps"], mean, label=config["label"],
                 color=config.get("color"), linewidth=2)
        plt.fill_between(df["steps"], mean - std, mean + std,
                         color=config.get("color"), alpha=0.1)
        plotted = True

    if not plotted:
        print("No data plotted. Check that the CSV files contain numeric data.")
        return

    plt.title("Impact of LLM-Imagined Exploration on Sample Efficiency", fontsize=14)
    plt.xlabel("Real-World Interaction Steps", fontsize=12)
    plt.ylabel(f"Episode Return (rolling avg, w={window})", fontsize=12)
    plt.legend(loc="lower right")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.tight_layout()
    plt.savefig(output_name, dpi=300)
    plt.show()


def backup_and_clear_data(target_dirs=("models/", "datasets/")):
    """Archive existing models and datasets, then recreate empty directories."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_root = f"backups/experiment_{timestamp}"
    print(f"Archiving existing data to {backup_root}...")

    for folder in target_dirs:
        if os.path.exists(folder):
            os.makedirs(backup_root, exist_ok=True)
            shutil.move(folder, os.path.join(backup_root, os.path.basename(folder.strip("/"))))
            os.makedirs(folder, exist_ok=True)
    print("Previous data archived; starting from a clean slate.")

## Training loop

Each iteration interacts with the real environment, optionally generates and
filters dreams, performs an offline update, and checkpoints the policy.

In [ ]:
def collect_dreams(params, max_retries=5, retry_delay=1):
    """Generate several batches of dreams, then keep the global top-k by reward.

    Returns True if any dreams were generated and stored, otherwise False.
    """
    all_dreams = {}
    all_rewards = {}

    for batch in range(params["num_sample_iterations"]):
        for attempt in range(max_retries):
            try:
                dreams = generate_dreams(
                    hist_data_path=params["hist_data_path"],
                    num_historical=params["num_historical"],
                    num_samples_generated=params["num_samples_generated"],
                )
                if dreams is None:
                    raise ValueError("LLM returned no transitions")
                rewards = evaluate_dreams(dreams)
                if rewards is None:
                    raise ValueError("LLM returned no reward estimates")

                # Namespace keys so batches across iterations never collide.
                for key in dreams:
                    if key in rewards:
                        unique_key = f"iter_{batch}_sample_{key}"
                        all_dreams[unique_key] = dreams[key]
                        all_rewards[unique_key] = rewards[key]
                print(f"Dream batch {batch + 1} collected on attempt {attempt + 1}.")
                break
            except Exception as e:
                print(f"Dream batch {batch + 1}, attempt {attempt + 1} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)

    if not all_dreams:
        return False

    print(f"Collected {len(all_dreams)} candidates; selecting top {params['top_k']}.")
    append_dreams(all_dreams, all_rewards, top_k=params["top_k"])
    return True


def train_agent(model, params):
    """Run one full experiment: alternating online interaction and offline updates."""
    experiment_name = "llm" if params["use_llm"] else "baseline"
    results_path = f"datasets/results_{experiment_name}.csv"
    logger = LoggerCallback(hist_csv_path=params["hist_data_path"], results_path=results_path)

    num_iterations = params["total_steps"] // params["num_steps_per_iteration"]

    for iteration in range(num_iterations):
        print(f"\n--- Iteration {iteration + 1}/{num_iterations} ({experiment_name.upper()}) ---")

        # Phase 1: online interaction with the real environment.
        model.learn(
            total_timesteps=params["num_steps_per_iteration"],
            callback=logger,
            reset_num_timesteps=False,
        )
        logger.save_to_csv()

        # Phase 2: offline learning, optionally augmented with dreams.
        used_dreams = collect_dreams(params) if params["use_llm"] else False
        if used_dreams:
            train_on_dreams(model, hist_data_path=params["hist_data_path"],
                            gradient_steps=params["gradient_steps"],
                            batch_size=params["batch_size"])
        else:
            print("Training on historical data only.")
            train_historical(model, params["hist_data_path"],
                             gradient_steps=params["gradient_steps"],
                             batch_size=params["batch_size"])

        # Phase 3: checkpoint the policy.
        save_path = f"models/dqn_{experiment_name}_{model.num_timesteps}_steps"
        model.save(save_path)
        print(f"Saved checkpoint: {save_path}")

## Run the experiments

Trains the baseline DQN and the LLM-augmented agent in turn, resuming from the
latest checkpoint of each if one exists, then plots the comparison.

In [ ]:
params = {
    "hist_data_path": "datasets/historical_data.csv",  # overridden per experiment below
    "num_historical": 30,           # real transitions shown to the LLM as context
    "num_samples_generated": 10,    # dreams requested per LLM call
    "num_sample_iterations": 10,    # LLM calls per offline phase (-> 100 candidates)
    "top_k": 50,                    # dreams kept after reward filtering
    "gradient_steps": 1000,         # offline gradient steps per iteration
    "batch_size": 64,
    "total_steps": 1000,            # real interaction steps per experiment
    "num_steps_per_iteration": 1000,
    "use_llm": False,               # set per experiment in the loop below
    "restart_fresh": False,         # archive existing models/datasets before running
}

configs = [
    {"path": "datasets/results_baseline.csv", "label": "Baseline (DQN)", "color": "tab:red"},
    {"path": "datasets/results_llm.csv",
     "label": "LLM-Augmented (LLAMBO-RL)", "color": "tab:blue"},
]

In [ ]:
if params["restart_fresh"]:
    backup_and_clear_data()

for use_llm in (False, True):
    experiment_name = "llm" if use_llm else "baseline"
    params["use_llm"] = use_llm
    params["hist_data_path"] = f"datasets/historical_data_{experiment_name}.csv"

    os.makedirs("models", exist_ok=True)
    existing = [f for f in os.listdir("models") if experiment_name in f]
    if existing:
        # Filenames follow 'dqn_<name>_<steps>_steps.zip'; resume from the latest.
        latest = max(existing, key=lambda f: int(f.split("_")[2]))
        print(f"Resuming {experiment_name} model from {latest}.")
        model = DQN.load(os.path.join("models", latest), env=vec_env)
    else:
        print(f"Creating new {experiment_name} model.")
        model = DQN("MlpPolicy", vec_env, verbose=1)

    train_agent(model, params)

In [ ]:
plot_learning_comparison(configs)